[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance2_exercices.ipynb)

# Séance 2.2 — Nettoyer des données réelles

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les sept défauts classiques d'un fichier réel
- convertir du texte en nombres et en dates
- traiter les valeurs manquantes en connaissance de cause
- supprimer les doublons et écarter les valeurs aberrantes
- construire un pipeline de nettoyage qu'on peut rejouer

## Comment ça marche

Chaque exercice se termine par une cellule de **vérification** qui vous dit immédiatement si votre réponse est bonne. Exécutez-la après avoir complété votre code.

Les `____` sont les trous à remplir. Tout le reste est déjà écrit.

> ⚠️ Si la vérification affiche `NameError`, c'est que la cellule au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la, puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")

print(sale.shape)
sale.head(3)

### Exercice 1 — Le diagnostic

> **Votre mission :**
> - Compter les lignes strictement dupliquées dans `nb_doublons`.
> - Compter les valeurs manquantes de `client_id` dans `nb_manquants`.

In [ ]:
nb_doublons = sale.____().sum()
nb_manquants = sale["client_id"].____().sum()

print(nb_doublons, "doublons |", nb_manquants, "manquants")

In [ ]:
verifier("1a - doublons", nb_doublons == 251, "duplicated() porte sur la ligne entiere")
verifier("1b - valeurs manquantes", nb_manquants == 429, "isna() puis .sum()")

### Exercice 2 — Dédoublonner (toujours en premier)

> **Votre mission :**
> - Créer `net` : `sale` sans les lignes dupliquées.
> - ⚠️ On dédoublonne **avant tout le reste**. Filtrer d'abord ferait perdre le compte de ce qu'on retire.

In [ ]:
net = sale.____().copy()

print(len(net))

In [ ]:
verifier("2 - apres dedoublonnage", len(net) == 5119,
         "5370 - 251 : la methode s'appelle drop_duplicates()")

### Exercice 3 — Écarter les ventes sans client

> **Votre mission :**
> - Retirer de `net` les lignes dont `client_id` est manquant.
> - Rappel : on le fait parce que notre question porte sur les **clients**. Pour une question sur le chiffre d'affaires total, ce serait une erreur.

In [ ]:
net = net.____(subset=["client_id"]).copy()

print(len(net))

In [ ]:
verifier("3 - lignes avec client", len(net) == 4712,
         "dropna(subset=[...]) cible une colonne precise")

### Exercice 4 — Le prix en nombre

> **Votre mission :**
> - Enlever le suffixe ` EUR`, remplacer la virgule par un point, convertir en nombre.
> - Remettre le résultat dans `net["prix"]`.
> - Puis vérifier qu'aucune valeur n'a été perdue : `nb_prix_perdus`.

In [ ]:
txt = net["prix"].str.replace(" EUR", "", regex=False)
txt = txt.str.replace(",", "____", regex=False)

net["prix"] = pd.____(txt, errors="coerce")
nb_prix_perdus = net["prix"].isna().sum()

print(net["prix"].dtype, "|", nb_prix_perdus, "valeurs perdues")

In [ ]:
verifier("4a - prix numerique", net["prix"].dtype == "float64",
         "pd.to_numeric convertit une colonne texte en nombres")
verifier("4b - aucune perte", nb_prix_perdus == 0,
         "si > 0, c'est qu'il reste du texte non converti dans la colonne")

### Exercice 5 — Les dates, correctement

> **Votre mission :**
> - Convertir `net["date"]` en vraies dates.
> - Le fichier mélange `14/11/2011` et `24-11-2011`, et il est au format **français**.
> - Puis mettre la date la plus ancienne dans `date_min`.

In [ ]:
net["date"] = pd.to_datetime(net["date"], format="____", dayfirst=____)
date_min = net["date"].min()

print(date_min)

In [ ]:
verifier("5 - date la plus ancienne", str(date_min)[:10] == "2010-12-01",
         "sans dayfirst=True, 01/12/2010 est lu comme le 12 janvier")

### Exercice 6 — Extraire le mois

> **Votre mission :**
> - Créer la colonne `mois` à partir de `date`.
> - Mettre le numéro du mois qui compte le plus de lignes dans `mois_top`.

In [ ]:
net["mois"] = net["date"].____.month
mois_top = net["mois"].value_counts().____()

print(mois_top)

In [ ]:
verifier("6 - mois le plus charge", mois_top == 10,
         ".dt.month sur une colonne de dates, puis value_counts().idxmax()")

### Exercice 7 — Uniformiser les catégories

> **Votre mission :**
> - Enlever les espaces autour et tout passer en minuscules.
> - Mettre le nombre de catégories restantes dans `nb_cat`.

In [ ]:
net["categorie"] = net["categorie"].str.____().str.____()
nb_cat = net["categorie"].nunique()

print(nb_cat)

In [ ]:
verifier("7 - categories uniformisees", nb_cat == 8,
         "il en reste beaucoup plus si vous n'avez fait que l'une des deux operations")

### Exercice 8 — Retours et aberrations

> **Votre mission :**
> - Compter les **retours** (`qte` strictement négatif) dans `nb_retours`.
> - Compter les quantités **aberrantes** (`qte` ≥ 10 000) dans `nb_aberrants`.
> - Puis créer `final` : les quantités strictement positives et inférieures à 10 000.

In [ ]:
nb_retours = len(net.query("____"))
nb_aberrants = len(net.query("qte >= 10000"))

final = net.query("qte > 0 ____ qte < 10000").copy()

print(nb_retours, "retours |", nb_aberrants, "aberrants |", len(final), "conservees")

In [ ]:
verifier("8a - retours", nb_retours == 107, "la condition est qte < 0")
verifier("8b - aberrants", nb_aberrants == 14, "la condition est qte >= 10000")
verifier("8c - lignes conservees", len(final) == 4591,
         "dans query() les conditions se combinent avec and")

### Exercice 9 — Le chiffre d'affaires nettoyé

> **Votre mission :**
> - Créer la colonne `ca` sur `final`, puis son total dans `ca_propre` (arrondi à 2 décimales).
> - Ce chiffre n'aurait été calculable **à aucun moment** avant le nettoyage : `prix` était du texte.

In [ ]:
final["ca"] = final["____"] * final["____"]
ca_propre = round(final["ca"].____(), 2)

print(ca_propre)

In [ ]:
verifier("9 - CA apres nettoyage", ca_propre == 111996.48,
         "verifiez que prix est bien numerique (exercice 4)")

### Exercice 10 — Le compte rendu

> **Votre mission :**
> - Calculer le **taux de perte** en pourcentage, arrondi à 1 décimale, dans `taux_perte`.
> - Formule : `100 * (1 - lignes_finales / lignes_initiales)`.
> - C'est le chiffre que vous présentez à votre responsable — pas « j'ai nettoyé les données ».

In [ ]:
taux_perte = round(100 * (1 - len(____) / len(____)), 1)

print("perte :", taux_perte, "%")

In [ ]:
verifier("10 - taux de perte", taux_perte == 14.5,
         "comparez le fichier final au fichier de depart sale")